# 12. 计算性能（章节总览）

> 原书：[《动手学深度学习 v2》第 12 章](https://zh-v2.d2l.ai/chapter_computational-performance/index.html)

深度学习中数据集和模型通常都很大，计算量随之巨大 → **计算性能至关重要**。
本章回答一个核心问题：**同一个模型，如何在不损失精度（甚至不改动模型本身）的前提下大幅缩短训练时间？**

本章主线可以概括为四步：

1. **换执行方式** —— 用编译替代逐行解释，消除 Python 前端开销（12.1）；
2. **别闲着** —— 前端与后端异步执行、多设备自动并行，让 CPU / GPU / 通信互不等待（12.2、12.3）；
3. **懂硬件** —— 理解内存层次与 CPU / GPU / 总线的带宽-延迟特征，才能定位真正瓶颈（12.4）；
4. **摊开算** —— 单卡不够就多卡：数据并行 → DDP 简洁实现 → 多机参数服务器（12.5–12.7）。

## 全章路线图

| 小节 | 主题 | 核心问题 | 关键词 |
| --- | --- | --- | --- |
| 12.1 | 编译器和解释器 | Python 解释器太慢怎么办？ | 符号式、混合式、`torch.compile` / `jit` |
| 12.2 | 异步计算 | CPU 为什么要等 GPU？ | 前端/后端、同步点、阻塞器 |
| 12.3 | 自动并行 | 多设备任务如何调度？ | 设备间并行、计算-通信重叠 |
| 12.4 | 硬件 | 真正的瓶颈在哪里？ | 内存层次、CPU/GPU、总线带宽 |
| 12.5 | 多GPU训练 | 单卡放不下 / 跑不快怎么办？ | 数据并行、all-reduce、梯度同步 |
| 12.6 | 多GPU的简洁实现 | 如何用最少的代码多卡训练？ | `DataParallel`、`DistributedDataParallel` |
| 12.7 | 参数服务器 | 如何扩展到多机？ | push/pull、键值存储、环同步 |

## 12.1 编译器和解释器

**两种编程范式：**

| | 命令式编程 | 符号式编程 |
| --- | --- | --- |
| 执行方式 | 逐行解释执行 | 先定义完整计算图，再编译执行 |
| 代表 | NumPy、PyTorch | Theano、TensorFlow 1.x |
| 优点 | 灵活、易调试、可穿插 Python 逻辑 | 运行前可见全局结构 → **算子融合、内存复用**等优化 |
| 缺点 | Python 前端开销大 | 调试困难、灵活性差 |

**混合式编程（hybridization）**：用命令式的方式写代码，运行时把（热的）计算固化成静态图再执行 —— 兼得两者优点。

PyTorch 中的三种对应物：

| API | 方式 | 特点 |
| --- | --- | --- |
| `torch.jit.trace` | 跟踪一次前向，记录实际执行过的算子 | 简单；**控制流会被"拍平"**（只保留本次走向） |
| `torch.jit.script` | 编译器解析源码（TorchScript 语言） | 保留 if/for 等控制流；需类型标注 |
| `torch.compile` | 编译整个模型（Dynamo + Inductor） | 更快更智能；对环境要求较高 |

**Sequential 的混合式** —— 整网直接编译：

```python
net = nn.Sequential(nn.Linear(20, 256), nn.ReLU(), nn.Linear(256, 10))
net = torch.compile(net)      # 或 torch.jit.script(net)
```

编译带来的收益：

1. 消除 Python 逐算子调度开销；
2. **算子融合**：如 Linear+ReLU 合成一个 kernel，少一次显存读写；
3. 为特定硬件生成专用内核。

> ⚠️ 混合式是"锦上添花"：当模型够大、计算本身已饱和时收益趋近于 0；只有当网络由很多小算子组成、前端开销占比高时收益显著。

## 12.2 异步计算

PyTorch 对 CUDA 操作采用**异步执行**：Python 前端把 kernel **压入队列**后立即返回、继续执行下一行，真正的计算由 GPU 后端按依赖顺序排队完成。前端因此可以"跑在前面"，持续喂任务给 GPU，不让它空闲。

```
Python 前端 :  [kernel1][kernel2][kernel3] ...        （不停入队）
GPU 后端    :         [kernel1][kernel2][kernel3]    （按依赖顺序执行）
```

**哪些时刻前端会真正等待（同步点 / 阻塞器）：**

- `tensor.item()`、`float(tensor)`：把值取回 CPU，必须等计算完成；
- `tensor.cpu()` / `print(tensor)`：跨设备拷贝；
- 显式调用 `torch.cuda.synchronize()`：等待**所有** kernel 执行完毕。

> 💡 **计时陷阱**：测 GPU 代码耗时时，若不先 `torch.cuda.synchronize()`，`time.time()` 测到的只是"入队耗时"（微秒级假象），不是真实计算时间。正确姿势：`synchronize() → 开始计时 → 计算 → synchronize() → 结束计时`。

**改进计算**：利用异步性做**重叠** ——
- CPU 端预处理 / 预取下一批数据，与 GPU 当前 batch 的计算重叠；
- 前端跑 Python 逻辑时，后端仍在算上一个任务。

异步不是让单个 kernel 变快，而是**不让任何部件闲着**。

## 12.3 自动并行

深度学习框架会**自动并行化**互不依赖的任务：把它们派发到不同设备（或同一设备的不同 stream）上同时执行，用户无需手动开线程。

典型场景 —— 两份数据、两张卡同时计算：

```python
x_gpu1 = x.to('cuda:1')     # 拷贝到卡 1
y_gpu2 = y.to('cuda:2')     # 拷贝到卡 2（与上面自动并行）
# 之后对两个结果在不同设备上计算，框架同样自动并行调度
```

要点：

- 依赖同一份数据的操作自动按序执行（框架维护依赖关系）；
- **互不依赖**的操作（不同设备、不同数据）自动并行；
- 需要手动等待某个结果时：触发一次同步点或 `torch.cuda.current_stream().synchronize()`。

**并行计算与通信**：多卡训练中最昂贵的往往不是计算而是**通信**（拷贝数据、同步梯度）。核心技巧是**计算-通信重叠**：上一批的梯度 all-reduce（通信）与下一批的前向传播（计算）同时进行 —— 这是 12.5 / 12.6 中 DDP 等实现的默认策略。

## 12.4 硬件

理解性能问题的根源需要一张硬件地图。

**内存层次**（越靠近处理器越快、越小、越贵）：

| 存储 | 典型延迟 | 典型带宽 |
| --- | --- | --- |
| 寄存器 | ~0.3 ns | — |
| L1 / L2 缓存 | ~1 / ~4 ns | 数十 GB/s |
| L3 缓存 | ~10 ns | — |
| 内存 (RAM) | ~100 ns | ~50 GB/s |
| NVMe SSD | ~100 µs | ~3 GB/s |
| 网络（同机房） | ~ms | 10~400 Gbps |

**CPU vs GPU：**

| | CPU | GPU |
| --- | --- | --- |
| 设计目标 | 低延迟：尽快完成单个任务 | 高吞吐：同时处理海量任务 |
| 核心数 | 几十核（强单核、复杂控制） | 数千~上万通道（SM） |
| 擅长 | 逻辑、分支、串行依赖 | 矩阵乘 / 卷积等规则稠密计算 |
| 缓存 / 显存 | 大缓存 | 高带宽显存 (GDDR6/HBM, ~1 TB/s) |

**网络与总线**（多卡 / 多机时的通信瓶颈）：

| 互连 | 带宽（约） |
| --- | --- |
| PCIe 4.0 x16 | ~32 GB/s |
| NVLink | ~600 GB/s |
| InfiniBand | 200~400 Gbps |

**实践启示：**

1. 训练瓶颈常常不在算力而在**数据搬运**（磁盘 → 内存 → 显存 → 显存间）；
2. 提高每字节搬运所换来的计算量（**算术强度**），例如融合算子、加大 batch；
3. 多卡扩展时，通信带宽（而不是 GPU 数量）往往决定加速比上限。

## 12.5 多GPU训练

**问题拆分的四种方式**（如何把工作摊到多张卡上）：

| 方式 | 做法 | 适用 |
| --- | --- | --- |
| 数据并行 | 每卡**完整模型副本**，各喂一部分数据 | 最常用，模型单卡放得下 |
| 模型并行 | 把**网络层**拆到不同卡（层间串行） | 单卡放不下模型 |
| 通道 / 权重拆分 | 把大矩阵按维切分（层内并行） | 超大矩阵 / 超大 embedding |
| 混合 | 以上组合（如流水线 + 数据并行） | 大规模训练 |

**数据并行的标准流程**（每一步迭代）：

1. 各卡持有同一份模型参数（初始一致）；
2. 把一个 batch 切成 $n$ 份，分发到各卡；
3. 各卡独立：前向 → 算 loss → 反向，得到**本地梯度** $\mathbf{g}_i$；
4. **all-reduce**：聚合梯度 $\bar{\mathbf{g}} = \frac{1}{n}\sum_i \mathbf{g}_i$，使每张卡都拿到相同的 $\bar{\mathbf{g}}$；
5. 各卡用 $\bar{\mathbf{g}}$ 各自更新参数 —— 初值与梯度一致，**参数始终保持同步**；
6. 进入下一轮迭代。

**数据同步**：聚合梯度的通信是主要开销。朴素做法（一台主卡收发）主卡带宽成瓶颈；**环同步（ring all-reduce）** 让每台机器只与相邻机器通信，通信量摊平为 $2(n-1)/n$ 倍参数大小，接近带宽最优。

**数据分发**：保证各卡拿到不同数据 —— PyTorch 用 `DistributedSampler` 按 rank 切分数据集。

> ⚠️ 数据并行把有效 batch size 变成 $n \times$ 单卡 batch size，学习率通常需要相应放大（线性缩放规则）。

## 12.6 多GPU的简洁实现

PyTorch 提供两个现成的数据并行封装：

| | `nn.DataParallel`（DP） | `DistributedDataParallel`（DDP） |
| --- | --- | --- |
| 进程模型 | 单进程多线程 | 每卡一个进程 |
| 通信方式 | 主卡收集 / 广播（瓶颈在主卡） | all-reduce 集合通信（均衡） |
| 启动方式 | 一行代码 | `torchrun` + 初始化进程组 |
| 官方态度 | 已不推荐 | **推荐**（更快、更稳、支持多机） |

**DP 用法**（几乎零成本）：

```python
net = nn.DataParallel(net, device_ids=range(4))
```

**DDP 标准骨架**：

```python
import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

dist.init_process_group('nccl')                  # ① 初始化进程组
local_rank = int(os.environ['LOCAL_RANK'])
torch.cuda.set_device(local_rank)

sampler = DistributedSampler(dataset)            # ② 按进程切分数据
loader = DataLoader(dataset, batch_size=64, sampler=sampler)

net = DDP(net.to(local_rank))                    # ③ 包装模型

for epoch in range(num_epochs):
    sampler.set_epoch(epoch)                     # ④ 保证每轮 shuffle 不同
    for X, y in loader:
        ...                                      # 正常训练，梯度自动 all-reduce
```

**网络初始化**：DDP 在构造时自动把 0 号进程的参数广播到所有进程，无需手动同步参数。

单机没有多卡时，也可用 `gloo` 后端在 CPU 上调试 DDP 流程。

## 12.7 参数服务器

把数据并行扩展到**多机**时，除 all-reduce 外还有另一种经典架构 —— **参数服务器（parameter server）**：

- **worker**：负责计算（前向 / 反向），持有数据分片；
- **server**：负责**存储与更新参数分片**（可多台，参数按 key 切成分片）。

交互通过两个原语完成：

- **push**：worker 把本地梯度发送给 server；
- **pull**：worker 从 server 拉取最新参数。

参数用**键值存储**（key-value store）管理：每层 / 每个参数块是一个 key，server 按 key 分摊存储与通信压力。

**环同步（ring synchronization）**：$n$ 台机器组成环，梯度分 $n$ 块沿环传递，每步只传相邻节点：

$$\text{总通信量} = \frac{2(n-1)}{n} \cdot |\mathbf{W}| \quad (\text{随 } n \text{ 增大趋近 } 2|\mathbf{W}|\text{，带宽最优})$$

**同步 vs 异步：**

| | 同步 | 异步 |
| --- | --- | --- |
| 行为 | 等所有 worker 提交梯度后统一更新 | worker 不等待，用当前参数继续算 |
| 优点 | 数值上等价于大 batch，收敛稳定 | 无等待，硬件利用率高 |
| 风险 | 慢 worker 拖累全局（straggler） | **梯度过期（stale）**可能伤收敛 |

现代实践：中小规模训练以 **DDP（同步 all-reduce）** 为绝对主流；异步参数服务器思想更多存在于超大规模推荐系统与分布式框架中。

## 小结：性能优化知识地图

| 性能问题 | 药方 | 对应小节 |
| --- | --- | --- |
| Python 前端调度开销大 | 编译 / 混合式（`torch.compile`、`jit.trace/script`） | 12.1 |
| CPU 与 GPU 互相等待 | 异步执行 + 正确的同步点 | 12.2 |
| 多设备任务串行执行 | 框架自动并行、计算-通信重叠 | 12.3 |
| 数据搬运慢、带宽瓶颈 | 理解内存层次、提高算术强度、融合算子 | 12.4 |
| 单卡放不下 / 训练太慢 | 数据并行（梯度 all-reduce） | 12.5 |
| 多卡代码太复杂 | `DistributedDataParallel` 一站式封装 | 12.6 |
| 需要多机扩展 | 参数服务器 / 环同步 / 键值存储 | 12.7 |

**一句话总结**：先让单卡跑满（编译 + 异步），再看硬件瓶颈（带宽），最后才是横向扩展（多卡多机）。

## 动手实验

两个不依赖 GPU 的实验，验证本章两个最核心的论断（在 `.venv` 内核中直接运行）：

1. **实验 1（对应 12.1 / 12.2）**：比较命令式（eager）与 `torch.jit.trace` 编译后的推理耗时，体会"混合式编程消除前端开销"；计时代码中演示 `torch.cuda.synchronize()` 的正确用法。
2. **实验 2（对应 12.5）**：从数值上验证数据并行的正确性 —— "把 batch 拆成两半分别求梯度再平均"与"全 batch 一次反向"得到的梯度**完全一致**（这正是 all-reduce 取平均的数学基础）。

In [ ]:
# 实验 1：命令式 vs 编译式（混合式编程）的推理耗时对比
# 对应 12.1：Python 前端有调度开销，编译后可消除部分开销
import time
import torch
import torch.nn as nn

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'


class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(200, 512), nn.ReLU(),
            nn.Linear(512, 512), nn.ReLU(),
            nn.Linear(512, 10))

    def forward(self, x):
        return self.net(x)


model = MLP().to(device)
x = torch.randn(256, 200, device=device)


def bench(fn, n=200):
    # 返回每次调用耗时（ms）；GPU 计时必须前后同步，否则测到的只是入队时间（12.2）
    for _ in range(10):                      # 预热
        with torch.no_grad():
            fn()
    if device == 'cuda':
        torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n):
        with torch.no_grad():
            fn()
    if device == 'cuda':
        torch.cuda.synchronize()             # 等所有 kernel 真正执行完
    return (time.perf_counter() - t0) / n * 1000


eager_ms = bench(lambda: model(x))

# torch.jit.trace：跟踪一次前向，把命令式代码固化成静态图（12.1 混合式）
with torch.no_grad():
    traced = torch.jit.trace(model.eval(), x)
scripted_ms = bench(lambda: traced(x))

print(f'device      : {device}')
print(f'eager       : {eager_ms:.3f} ms / batch')
print(f'jit.trace   : {scripted_ms:.3f} ms / batch')
print(f'加速比      : {eager_ms / scripted_ms:.2f}x')
print('输出一致    :', torch.allclose(model(x), traced(x), atol=1e-5))

# 可选：torch.compile（需要较新 PyTorch，Windows 下可能不可用，包在 try 中）
try:
    compiled = torch.compile(model)
    compiled_ms = bench(lambda: compiled(x))
    print(f'torch.compile: {compiled_ms:.3f} ms / batch, 加速比 {eager_ms / compiled_ms:.2f}x')
except Exception as e:
    print('torch.compile 不可用:', e)

In [ ]:
# 实验 2：数据并行的数值本质 —— 拆分 batch 各自求梯度，再 all-reduce 平均
# 对应 12.5：验证"各卡本地梯度的平均 == 全 batch 一次前向反向的梯度"
import torch
import torch.nn as nn

torch.manual_seed(0)

net = nn.Linear(4, 3, bias=True)     # 极简网络
X = torch.randn(32, 4)               # 全 batch：32 个样本
y = torch.randn(32, 3)
loss_fn = nn.MSELoss()

# ① 单卡（全 batch 一次计算）
net.zero_grad()
loss_fn(net(X), y).backward()
grad_full = net.weight.grad.clone()

# ② 模拟 2 张卡：各拿一半数据，独立求本地梯度
X_parts, y_parts = torch.chunk(X, 2), torch.chunk(y, 2)
grad_parts = []
for Xb, yb in zip(X_parts, y_parts):
    net.zero_grad()
    loss_fn(net(Xb), yb).backward()
    grad_parts.append(net.weight.grad.clone())   # 本卡"上传"的梯度

# ③ all-reduce：求和后取平均（等价于 DDP 的默认行为）
grad_avg = sum(grad_parts) / len(grad_parts)

print('全 batch 梯度 :', grad_full[0][:4].tolist())
print('平均后的梯度  :', grad_avg[0][:4].tolist())
print('最大绝对误差  :', f"{(grad_full - grad_avg).abs().max().item():.2e}")
print('数值等价      :', torch.allclose(grad_full, grad_avg, atol=1e-6))